# 01. 연구배경 및 문제정의

## (1) 연구배경 및 핵심문제

`1` GWAS를 통해 발굴된 공통변이(Common Variant)들은 `Complex diseases`의 유전력 중 일부만 설명할 수 있음. `Missing Heritability` 문제가 발생함.

`2` 이를 해결하고자 `Next-Generation Sequencing(NGS)` 기술이 등장했고, 희귀변이를 찾아내는 것 자체는 성공하였음.

`3` 그러나 희귀변이는 MAF가 극도록 낮다보니, `단일 변이 분석`(Single-Variant test)를 적용할 경우 통계적 검정력이 현저히 떨어짐.

`4` 이를 극복하기 위해 유전자나 특정영역 단위들로 변이들을 집계하여 검정하는 `Gene-level aggregation test`가 필수적으로 요구되었으며, 금번 리뷰에서는 이와 관련된 Method들을 소개함.

<details>
<summary> 용어 정리</summary>

`1` Complex diseases: 고혈압, 당뇨병, 암, 성 조숙증 등과 같이 유전자 카드 수십~수천 장과 식습관, 운동과 같은 생활 환경이 복잡하게 얽혀서 발생하는 질병, 즉 `유전자 + 환경`이 합쳐서 나타나는 질병들을 일컫는다.

`2` Missing Heritability(놓친 유전력): 전체 유전력(Overall Heritability)과 GWAS로 발굴된 설명력 간의 격차를 의미. 여기서 놓친 설명력이 회귀변이에 있으며, 이러한 설명력을 확보하기 위해 NGS 시퀀싱과 금일 리뷰할 방법론 연구들이 시작되었음.

`3` Next-Generation Sequencing(NGS): 대량 병렬 시퀀싱 기술로, 유전체 전체 또는 단백질을 코딩하는 Whole-Exome의 대량 DNA 염기서열 데이터를 고속으로 해독하는 분자생물학적 분석 플랫폼. 

`4` 단일 변이 분석(Single-Variant) test:
- genome상의 개별 변이와 질환 형질 간의 연관성을 1대1로 검정하는 전통적인 방법
- Common Variants 분석에서는 효과적이지만, Rare Variants의 경우 관측빈도가 낮아 대표본이 확보되지 않는 한 검정력이 심각하게 저해됨.

`5` Gene-level aggregation test
- 특정 유전자나 서열 구역(Genomic region) 내에 존재하는 Rare Variants 효과를 하나의 단위로 집계(Aggreation/Collapsing)하여 질환과의 연관성을 종합적으로 검증하는 기법임

</details>

<details>
<summary> 원문 </summary>

`1` GAAWS의 한계와 NGS 기술의 도래

> *"Genome-wide association studies (GWASs) have identified more than 1000 genetic loci associated with many human diseases and traits, yet common variants identified through GWASs often explain only a small proportion of trait heritability. The advent of massively parallel sequencing has transformed human genetics and has the potential to explain some of this missing heritability through identification of trait-associated rare variants."*

`2` 단일 변이 검정의 문제점

> *"These emerging sequencing technologies provide a rich opportunity to study the association between rare variants and complex traits... Single-variant tests are typically conducted to investigate associations of common variants and phenotypes; however the same approach has little power for testing for rare-variant effects because of their low frequencies and large numbers."*

> "*Standard methods used to test for association with single common genetic variants are underpowered for rare variants unless sample sizes or effect sizes are very large."*

`3` Gene-level aggregation test 등장

> *"Instead, the statistical development of rare-variant analysis has been focused on testing cumulative effects of rare variants in genetic regions or SNP sets, such as genes. These tests can be broadly classified as burden and nonburden tests."*

> *"The motivation behind this approach is that while a single variant is usually underpowered to detect statistical signals between case and control subjects, aggregating variants across a candidate gene might improve power."*

</details>

## (2) 초기 방법론의 한계

### a. Burden test(CAST, WST, CMC 등)의 한계

`1` 유전자 내 희귀변이들을 하나로 합쳐서 검정하는 방식

`2` 한계점

- 유전자 영역 내 모든 회귀변이가 `동일한 방향`(Unidirectional, 예: 모두 위험 변이)으로 작용한다는 비현실적인 가정에 의존함.

- 그러나 실제 유전체 영역에는 효과가 없는 `Non-causal variants`나 반대 작용을 하는 `Protective variants`가 혼재되어 있으면, 변이를 합치는 과정에서 노이즈가 유입되어 `Statistical Power`가 급격히 붕괴됨.

<details>
<summary>원문</summary>

> *A limitation for all these burden tests is that they implicitly assume that all rare variants influence the phenotype in the same direction and with the same magnitude of effect... Hence, collapsing across all variants is likely to introduce substantial noise into the aggregated index, attenuate evidence for association, and result in power loss.*

</details>

### b. Non-Burden Test(C-alpha test 등)

`1` 변이들을 하나로 합치지 않고 변이 빈도와 분산을 비교하여, 변이 효과의 방향성에 상관없이 검정하는 방식.

`2` 한계점

- 변이 방향성에는 강건(Robust)하지만, Population stratification 보정을 위한 **Covariates**(예: Principal Components)를 모델에 쉽게 포함하기 어렵고,

- Permutation을 통해 $p$-value를 계산해야 해서 Genome-wide scale 분석 시 계산 비용이 비현실적으로 큼.

<details>
<summary> 원문 </summary>

> *"The recently proposed C-alpha test is a non-burden-based test and is hence robust to the direction and magnitude of effect... Despite these attractive features, the C-alpha test does not allow for easy covariate adjustment, such as for controlling population stratification... [and] uses permutation to obtain a p value... which is computationally expensive for whole-genome experiments."*

</details>

---

# 02. Method & Key Results

## (1) 방법론 발전 과정

### step1. SKAT

#### a. 앞선 방법론의 한계점

`1` 모든 변이가 한 방향으로 흐른다. (`Burden test`)

`2` 변이들에 동일 방향성 가정은 없으나 공변량 보정이 어렵고 계산량이 크다. (`Non-Burden test`)

<details>
<summary>원문</summary>

> *A limitation for all these burden tests is that they implicitly assume that all rare variants influence the phenotype in the same direction and with the same magnitude of effect... Hence, collapsing across all variants is likely to introduce substantial noise into the aggregated index, attenuate evidence for association, and result in power loss.*

> *Despite these attractive features, the C-alpha test does not allow for easy covariate adjustment, such as for controlling population stratification... The C-alpha test also uses permutation to obtain a p value... which is computationally expensive for whole-genome experiments."*

</details>

#### b. 해결책

`1` 이를 해결하기 위해, 변이들을 하나로 뭉뚱그려 합치지 않고, 개별 변의의 효과 회귀계수 $\beta_j$가 평균 0, 분산 $w_j^2 \tau$인 임의의 분포를 따른다고 가정한 **Variance-component score test**를 적용함

<details>
<summary> Methodology</summary>

`1` 기본 모형

- $\beta_j$를 평균 0을 가지는 임의 변수로 취급 $\to\,\, \beta_j \sim(0, \tau w_j)$
    - $\tau$: 분산성분(Variance component): 영역 전체에 미치는 효과 `총량`
    - $w_j$: $j$번째 변이에 대한 사전 가중치.

- Score test로 전환 $\to$ covariate만 포함된 Null Model을 단 1회만 적합시키면 되므로, GWAS 분석 시 `계산 속도가 빨라짐`

$$H_0: \tau = 0 \quad \text{vs} \quad H_1: \text{not } H_0$$

<details>
<summary>원문</summary>

> *"To increase the power, SKAT tests $H_0$ by assuming each $\beta_j$ follows an arbitrary distribution with a mean of zero and a variance of $w_j \tau$... which can be conveniently tested with a variance-component score test... A key advantage of the score test is that it only requires fitting the null model...*

</details>

- Continuous trait: $y_i = \alpha_0 + \alpha' X_i + \beta' G_i + \epsilon_i$ (Equation 1)

- Dichotomous trait: $\text{logit}\,\, P(y_i = 1) = \alpha_0 + \alpha' X_i + \beta' G_i$ (Equation 2)

`2` 확장 모형(Semiparametric Models for Epistasis)

$$H_0: f(G) = 0 \quad \text{vs} \quad H_1: \text{not } H_0$$

- Continuous trait: $y_i = \alpha_0 + \alpha' X_i + f(G_i) + \epsilon_i$ (Equation 4)

- Dichotomous trait: $\text{logit}\,\,P(y_i = 1) = \alpha_0 + \alpha' X_i + f(G_i)$ (Equation 5)

> 선형 항 $\beta' G_i$를 임의의 함수 $f(G_i)$로 대체함으로써, 단일 변이 효과뿐만 아니라 희귀변이 $\times$ 희귀변이 또는 흔한변이 $\times$ 희귀변이 간의 복잡한 상위성 효과(Epistatic effects) 및 비선형적 상호작용까지 유연하게 모형화할 수 있음.

<details>
<summary>원문</summary>

> *An attractive feature of SKAT is the ability to model the epistatic effects of sequence variants on the phenotype within the flexible kernel machine regression framework. To do so, we replace $G_i'\beta$ by a more flexible function $f(G_i)$ in the linear and logistic models (1) and (2) where $f(G_i)$ allows for rare variant by rare variant and common variant by rare-variant interactions.*

<details>

<summary> 각 기호 의미 </summary>

- $y_i$: $i$번째 개인의 표현형(반응변수라고 생각, Phenotypes)

- $\alpha_0$: 절편(intercept)

- $\boldsymbol{\alpha}$: 공변량 $\mathbf{X_i}$(나이$\cdot$성별)의 효과 계수 벡터-고정 효과

- $\beta$: $p$개의 변이 각각이 반응변수(Phenotypes)에 미치는 효과 크기(계수, 회귀계수)벡터 $\to$ 검정대상

- $G_{i}$: $i$번째 사람이 몇 개의 Minor allele를 가졌는지`(0,1,2)`

|염색체1|염색체2|Minor allele 수($\boldsymbol{G}_{ij}$)|
|:---:|:---:|:---:|
|Major|Major|0|
|Major|Minor|1|
|Minor|Major|1|
|Minor|Minor|2|

</details>

</details>

`3` 검정통계량 $Q$ & 가중치($w_j$)

$$Q = (y - \hat{\mu})'K (y - \hat{\mu}) = \sum_{j=1}^p w_j^2 S_j^2$$

$$\sqrt{w_j} = \text{Beta}(\text{MAF}_j, 1, 25)$$

- $K = G W W G'$: 샘플 간 유전적 유사도를 측정하는 Kernel 행렬.

- 개별 변이 Score 통계량 $S_j$ 제곱의 가중 합산 형태로 $Q$가 형성되며, MAF가 낮은 희귀변이에 더 큰 가중치를 부여합니다.

- 귀무가설 하에서 $Q$는 Mixture of Chi-square 분포를 따르며, Davies method로 Permutation 없이 즉시 유의확률을 산출

</details>

#### c. 주요 결과

`1` Type 1 Error Rate

- Continuous traits에서는 유의수준인 $\alpha = 10^{-6}$에서도 Type I error을 부풀림 없이 정확하게 통제.

- 특이사항: Dichotomous traits 및 표본이 적은 경우에는 Asymptotic $p$-value 계산 특성상 $p$-value가 Deflated으로 산출되는 경향을 보임

<details>
<summary>Table 1-(1)</summary>

<center><img src = '02. table1.png' width = 500></center>

</details>

<details>
<summary>Table 1-(2)</summary>

<center><img src = '02. table1(2).png' width = 500></center>

</details>

`2` Statistical Power 비교

- $\beta$가 50대 50으로 섞여 있을 때 기존 Burden test들(W, N, C)의 Power은 0.1\~0.2 수준에 그치는 반면, **SKAT** 은 표본이 늘어남에 따라 0.7 이상으로 치솟는 모습을 보임

<details>
<summary>Figure 1</summary>

<center><img src = '02. fig1.png' width = 700></center>

</details>

<details>
<summary>Table 2</summary>

<center><img src = '03. table2.png' width = 500></center>

</details>

`3` 실데이터 검증

- 자료: *Analysis of the Dallas Heart Study Sequencing Data*

- DHS 데이터 3,476명의 *ANGPTL3, ANGPTL4, ANGPTL5* 유전자 내 93개 희귀변이 분석 결과를 수록한 표

- 표의 두 번째 행인 Dichotomized TG level 열을 보면 **SKAT의 $p$-value가 $1.3 \times 10^{-4}$** 로 명시되어 있으며, 옆에 적힌 CAST ($3.2 \times 10^{-2}$)나 Weighted Sum ($3.1 \times 10^{-3}$)보다 수치가 훨씬 작아(유의성이 더 높아) 우수한 검정력을 보임

<details>
<summary>Table 4</summary>

<center><img src = '04. table4.png' width = 900></center>

</details>

`4` 계산 효율성

- 원인 변이 비중이 높을수록, 효과 크기(OR)가 클수록 필요 표본 수가 지수적으로 감소하며, 유병률이 낮을수록 환자군 내 원인 변이 집적(Enrichment) 효과로 인해 필요 표본 수가 줄어드는 경향을 정량적으로 입증함.

<details>
<summary>Table 3</summary>

<center><img src = '05. table3.png' width = 900></center>

</details>

<details>
<summary>Figure 3</summary>

<center><img src = '03. fig2.png' width = 700></center>

- 표본 크기 $n = 2,000 \sim 10,000$명 구간에서 **SKAT의 이론적 추정 공식으로 그려낸 검정력 곡선(Theoretical Power, 검은선)** 과 **1,000회 시뮬레이션으로 얻은 실측 곡선(Empirical Power, 빨간선)** 을 직접 대조한 결과, 두 곡선이 **완벽하게 오버랩(Overlap)** 된 것으로 보임.

</details>

<details>
<summary>Figure 3</summary>

<center><img src = '04. fig3.png' width = 500></center>

</details>

#### d. 한계점


`1` 원인 변이 비율이 높고 단방향(Unidirectional)인 경우의 파워 열세

`2` 소표본($n < 1,000$) 케이스-컨트롤 연구 시 $p$-value Deflation이 발생함.